In [1]:
genes = Gene.objects.all()

In [2]:
import json

In [3]:
nodesjson = json.load(open('./static/visualization/Analysis/nodes.json'))

In [4]:
g = {x.orf: x for x in genes}

In [5]:
i = 0
for n in nodesjson['nodes']:
    n['isdu'] = g[n['orf']].feature_qualifier == 'Dubious'
    n['aliases'] = g[n['orf']].aliases
    if n['isdu']: i += 1

In [6]:
def _dump_clean_json(obj, f):
    with open(f, 'wb') as out:
        out.write(json.dumps(obj).replace(' ', ''))

In [7]:
_dump_clean_json(nodesjson, './static/visualization/Analysis/nodes.json')

In [1]:
from numpy import isnan

In [2]:
bud3 = Gene.objects.get(name='BUD3')

In [2]:
bud3s = bud3.strain_set.all()

In [20]:
x = bud3s[1]

In [25]:
bud3s[0].as_array.all(), bud3s[1].as_array.all()

([<Dataset: Analysis>], [<Dataset: Analysis>])

In [115]:
import pandas as p
import operator

In [4]:
def format_allele_col(orf, name, strainid, allele):
    strainid = strainid.lower()
    suffix = 'damp' in strainid and '_damp' or ''
    allele_col = (allele or name or orf).lower()
    
    if 'ts' not in strainid and 'damp' not in strainid:
        suffix = u'\u0394'
    
    if suffix:
        allele_col = '%s%s' % (allele_col, suffix)
    
    return allele_col

In [5]:
ONLY = (
    'strain__gene__orf',
    'strain__gene__name',
    'strain__boonelab_id',
    'strain__allele',
)

In [6]:
ds = Dataset.objects.all()[0]
arrays = [(strain[0], format_allele_col(*strain)) for strain in ds.arrays.through.objects.filter(dataset=ds).order_by('id').values_list(*ONLY)]
queries = [(strain[0], format_allele_col(*strain)) for strain in ds.queries.through.objects.filter(dataset=ds).order_by('id').values_list(*ONLY)]

In [99]:
scores = p.DataFrame()
for data in ds.data.filter(strain__in=bud3.strain_set.all()).select_related('strain__gene'):
    s = data.strain

    if data.type == StrainData.TYPE_QUERY:
        scores_axis = arrays
    elif data.type == StrainData.TYPE_ARRAY:
        scores_axis = queries

    scores = scores.append(p.DataFrame({'target': scores_axis, 'score': data.scores, 'pval': data.pvalues}), ignore_index=True)

In [128]:
scores = scores[scores.pval < 0.05].sort(['target', 'pval']).reindex(columns=['target', 'pval', 'score'])

In [129]:
grp = scores.groupby('target')

In [133]:
x = grp.filter(lambda x: len(x) < 2 or not operator.xor(*(x.score < 0))).groupby('target').first()

In [8]:
x

NameError: name 'x' is not defined

In [66]:
ds = Dataset.objects.all()[0]

common = []

for sd in ds.data.filter(strain__in=bud3.strain_set.all()).select_related('strain__gene'):
    print sd.strain
    if sd.type == 'Q':
        data = zip(sd.scores, sd.pvalues, ds.arrays.through.objects.filter(dataset=ds).order_by('id').select_related('strain__gene'))
    else:
        data = zip(sd.scores, sd.pvalues, ds.queries.through.objects.filter(dataset=ds).order_by('id').select_related('strain__gene'))
    
    data = list(filter(lambda x: not isnan(x[0]), data))
    common += data
    
    for sc, pv, st in sorted(data, key=lambda x: x[0])[:20]:
        print sc, pv, st.strain
    print 'FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFf'

YCL014W (BUD3) - sn1239
-0.1973 0.03287 YLR316C (TAD3) - tad3-5001 - tsa1150
-0.182 0.05007 YFL039C (ACT1) - act1-105 - tsa909
-0.1575 0.1208 YDR123C (INO2) - dma920
-0.1479 0.1849 YHR059W (FYV4) - dma2115
-0.1386 0.07967 YOR122C (PFY1) - pfy1-4 - tsa232
-0.137 0.2578 YGR074W (SMD1) - smd1-1 - tsa748
-0.1367 0.1468 YJR076C (CDC11) - cdc11-5 - tsa84
-0.1265 0.1134 YDR182W (CDC1) - cdc1-1 - tsa31
-0.1228 0.279 YMR043W (MCM1) - mcm1-5001 - tsa1134
-0.1204 0.1322 YKL048C (ELM1) - dma2781
-0.1084 0.2051 YKL006W (RPL14A) - dma2739
-0.1081 0.2939 YGR002C (SWC4) - swc4-4 - tsa895
-0.1054 0.2095 YDR168W (CDC37) - cdc37-ts - tsa315
-0.1049 0.2157 YKL022C (CDC16) - cdc16-1 - tsa883
-0.1047 0.1879 YHR065C (RRP3) - rrp3-5001 - tsa1261
-0.1043 0.1174 YDR083W (RRP8) - dma882
-0.1026 0.2346 YOL102C (TPT1) - tpt1-5001 - tsa1264
-0.1019 0.1873 YML130C (ERO1) - ero1-1 - tsa203
-0.1011 0.0876 YOR122C (PFY1) - pfy1-13 - tsa233
-0.1003 0.2022 YDL207W (GLE1) - gle1-4 - tsa920
FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFF

In [7]:
from django.db import transaction

In [6]:
genes = {g.primary_sgdid: g for g in Gene.objects.all()}
len(genes), Gene.objects.all().count()

(7153, 7153)

In [8]:
with transaction.atomic():
    for t in Term.objects.using('oldcellmap').all():
        thisterm = Term.objects.get(pk=t.pk)
        for g, in t.genes.values_list('primary_sgdid'):
            thisterm.genes.add(genes[g])

In [3]:
a = Annotation.objects.get(pk=17) 

In [10]:
for term in a.term_set.all():
    for s in term.strains.select_related():
        if s.gene.name == 'PAB1':
            print(term, s.label())

(<Term: nuclear nucleus protein export pore>, u'pab1-5002')
(<Term: nuclear nucleus protein export pore>, u'pab1-5004')
(<Term: regulation biosynthetic RNA transcription positive>, u'pab1-5003-supp1')
